In [0]:
df = spark.table(
    "proyecto_final_mlops.telco_database.churn_dataset_training_data"
)

display(df.limit(10))

CustomerID,Age,Gender,Tenure,Usage_Frequency,Support_Calls,Payment_Delay,Subscription_Type,Contract_Length,Total_Spend,Last_Interaction,Churn
2,30,Female,39,14,5,18,Standard,Annual,932.0,17,1
3,65,Female,49,1,10,8,Basic,Monthly,557.0,6,1
4,55,Female,14,4,6,18,Basic,Quarterly,185.0,3,1
5,58,Male,38,21,7,7,Standard,Monthly,396.0,29,1
6,23,Male,32,20,5,8,Basic,Monthly,617.0,20,1
8,51,Male,33,25,9,26,Premium,Annual,129.0,8,1
9,58,Female,49,12,3,16,Standard,Quarterly,821.0,24,1
10,55,Female,37,8,4,15,Premium,Annual,445.0,30,1
11,39,Male,12,5,7,4,Standard,Quarterly,969.0,13,1
12,64,Female,3,25,2,11,Standard,Quarterly,415.0,29,1


In [0]:
total = df.count()
unicos = df.dropDuplicates().count()

print("Registros totales:", total)
print("Registros únicos:", unicos)
print("Duplicados:", total - unicos)

Registros totales: 440832
Registros únicos: 440832
Duplicados: 0


In [0]:
from pyspark.sql.functions import count

duplicados_cliente = (
    df.groupBy("CustomerID")
      .agg(count("*").alias("cantidad"))
      .filter(col("cantidad") > 1)
)

print("CustomerID duplicados:", duplicados_cliente.count())

display(duplicados_cliente.limit(20))

CustomerID duplicados: 0


CustomerID,cantidad


In [0]:
from pyspark.sql.functions import round

total = df.count()

churn_distribution = (
    df.groupBy("Churn")
      .count()
      .withColumn(
          "porcentaje",
          round((col("count") / total) * 100, 2)
      )
      .orderBy("Churn")
)

display(churn_distribution)

Churn,count,porcentaje
0,190833,43.29
1,249999,56.71


In [0]:
numeric_cols = [
    "Age",
    "Tenure",
    "Usage_Frequency",
    "Support_Calls",
    "Payment_Delay",
    "Total_Spend",
    "Last_Interaction"
]

display(
    df.select(numeric_cols).summary()
)

summary,Age,Tenure,Usage_Frequency,Support_Calls,Payment_Delay,Total_Spend,Last_Interaction
count,440832,440832,440832,440832,440832,440832,440832
mean,39.37315349157956,31.25633574695122,15.80749355763647,3.6044366107723578,12.965721635452962,631.6162227787628,14.480867995063878
stddev,12.442369490866627,17.255727385042206,8.586241645186686,3.070217877688797,8.258062517501818,240.80300112056852,8.59620768302461
min,18,1,1,0,0,100.0,1
25%,29,16,9,1,6,480.0,7
50%,39,32,16,3,12,661.0,14
75%,48,46,23,6,19,830.0,22
max,65,60,30,10,30,1000.0,30


In [0]:
from pyspark.sql.functions import col

categorical_cols = [
    "Gender",
    "Subscription_Type",
    "Contract_Length"
]

for c in categorical_cols:
    print(f"\n--- {c} ---")
    display(
        df.groupBy(c)
          .count()
          .orderBy(col("count").desc())
    )


--- Gender ---


Gender,count
Male,250252
Female,190580



--- Subscription_Type ---


Subscription_Type,count
Standard,149128
Premium,148678
Basic,143026



--- Contract_Length ---


Contract_Length,count
Annual,177198
Quarterly,176530
Monthly,87104
